# Nightingale: XGBoost's seeds 43–46 (`docs/05` §6, amendment 3)

The protocol binds every retrained XGBoost to five seeds, **42, 43, 44, 45 and 46**, because its
row and column subsampling make it stochastic. The seed-42 model stays the one used everywhere; the
other seeds only show how much a result depends on the seed ("for how many of the five seeds the
verdict would also hold"). Seed 42 of each arm already exists (EXP-004, EXP-018). This notebook
trains the other four, on Colab, where the owner chose to run it (decision D-7):

| Arm | Training rows | Encoder | Seed 42 is |
|---|---|---|---|
| **B1-XGB** | the train split, as EXP-004 | 607 columns (`3a0d5a5e01d7f427`) | EXP-004's model |
| **B1-XGB+aug** | plus one masked copy of every patient | the same | EXP-018's |
| **B1-XGB′+aug** | the same | with the "asked" channel (`ddd14019c66eff10`) | EXP-018's |

The seed sets XGBoost's subsampling, its 10% early-stopping holdout and, for `+aug`, the masked
copies. Each run is `scripts/train_baselines.py`, unchanged, so each also refits B0 and logistic
regression: the plain logistic regression is deterministic and must come out as EXP-004's, and the
`+aug` logistic regressions vary with the seed (the errata's item 5); they are kept, labelled, and
used for nothing yet. As a check, **seed 42 of the plain arm is also refitted**: it should reproduce
EXP-004's validate scores. Validate is scored at full evidence here; the laptop scores the 50% and
25% levels afterwards on the recorded masks. The test split is never downloaded.

**Before you start:** the notebook asks for a GPU (T4); XGBoost uses it if it is there. If Colab
offers only a CPU, accept it: the run is just slower.

**Then:** *Runtime → Run all*. It takes about **50–60 minutes**: some 20 downloading and decoding
the 670 MB `train.csv`, then about 8 minutes per seed. Keep the tab open. At the end your browser
downloads **`nightingale_b1_seeds.zip`**. If the session drops, re-run: finished runs are skipped.

**Keep this notebook and its outputs private.** Models trained on DDXPlus are never published
(`docs/11` §4). Nothing here needs a password or a token.

In [ ]:
# The commit Claude gave you. "master" works too: run.json records the exact commit either way.
COMMIT = "master"

In [ ]:
import os

if not os.path.exists("/content/nightingale"):
    !git clone -q https://github.com/HarshRohila02/nightingale.git /content/nightingale
%cd /content/nightingale
!git checkout -q {COMMIT}
!git log --oneline -1
# XGBoost 2.1 is the version on the owner's laptop, so the laptop can load the trained models.
!pip install -q "xgboost~=2.1" "scikit-learn~=1.5"

In [ ]:
# DDXPlus from Hugging Face, at the snapshot the laptop uses (docs/11 §4).
# test.csv is never fetched before Phase 4 (docs/05 §2).
from huggingface_hub import hf_hub_download

for name in ["release_evidences.json", "release_conditions.json", "train.csv", "validate.csv"]:
    hf_hub_download(
        "aai530-group6/ddxplus",
        name,
        repo_type="dataset",
        revision="2ad986acc1ec62fb4a94171acc43f4fdd5bfde53",
        local_dir="data/raw/ddxplus",
    )
!ls -lh data/raw/ddxplus

In [ ]:
# Decode the vocabulary, then keep the 13 chest-pain conditions of each split (task 1a).
!python scripts/decode_ddxplus.py
!python scripts/build_ddxplus_chestpain.py --split train
!python scripts/build_ddxplus_chestpain.py --split validate

In [ ]:
import shutil
import subprocess
from pathlib import Path

DEVICE = "cuda" if shutil.which("nvidia-smi") else "cpu"
print("XGBoost runs on", DEVICE)

ARMS = {
    "b1": [],                                         # B1-XGB: EXP-004's rows and encoder
    "b1_aug": ["--augment"],                          # B1-XGB+aug
    "b1_asked_aug": ["--asked-channel", "--augment"], # B1-XGB′+aug
}
# Seed 42 of the plain arm is a reproducibility check against EXP-004, never a replacement.
RUNS = [("b1", 42)] + [(arm, seed) for seed in (43, 44, 45, 46) for arm in ARMS]


def run(arm, seed):
    out = Path("models/b1_seeds") / arm / f"seed{seed}"
    if (out / "run.json").exists():
        print(f"{arm} seed {seed}: already done, skipped")
        return
    print(f"=== {arm}, seed {seed} ===", flush=True)
    command = ["python", "scripts/train_baselines.py", "--device", DEVICE, "--seed", str(seed),
               *ARMS[arm], "--out", str(out)]
    subprocess.run(command, check=True)


for arm, seed in RUNS:
    run(arm, seed)

In [ ]:
# One zip to bring back: every bundle (models, metrics, run.json). No patient rows are in it.
import shutil
from google.colab import files

archive = shutil.make_archive("/content/nightingale_b1_seeds", "zip", "models/b1_seeds")
!du -sh models/b1_seeds && ls models/b1_seeds/* && ls -lh {archive}
files.download(archive)